In [1]:
# ============================================================
# Regresión Lineal con winequality-red.csv
# ============================================================

# 1. Importar librerías necesarias para cargar datos, entrenar el modelo y evaluar resultados.
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import statsmodels.api as sm

In [2]:
# 2. Cargar el archivo CSV usando el delimitador por defecto (coma).
df = pd.read_csv("winequality-red.csv")

In [3]:
# 3. Revisar rápidamente el tamaño del dataset y las primeras filas.
print("Tamaño del dataset:", df.shape)
print(df.head())

# 4. Definir la variable objetivo que queremos predecir: la calidad del vino.
target = "quality"

# 5. Separar las variables predictoras X y la variable objetivo y.
X = df.drop(columns=[target])
y = df[target]

Tamaño del dataset: (1599, 12)
   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.4              0.70         0.00             1.9      0.076   
1            7.8              0.88         0.00             2.6      0.098   
2            7.8              0.76         0.04             2.3      0.092   
3           11.2              0.28         0.56             1.9      0.075   
4            7.4              0.70         0.00             1.9      0.076   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 11.0                  34.0   0.9978  3.51       0.56   
1                 25.0                  67.0   0.9968  3.20       0.68   
2                 15.0                  54.0   0.9970  3.26       0.65   
3                 17.0                  60.0   0.9980  3.16       0.58   
4                 11.0                  34.0   0.9978  3.51       0.56   

   alcohol  quality  
0      9.4        5  
1      9.8 

In [4]:
# 6. Eliminar columnas identificadoras que no aportan valor predictivo directo al modelo.
# (En este dataset no hay columnas de id, pero dejamos la estructura segura con errors='ignore')
X = X.drop(columns=["id", "Id", "Order"], errors="ignore")

# 7. Identificar columnas numéricas y categóricas para tratarlas de forma diferente.
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = X.select_dtypes(include=["object"]).columns

# 8. Rellenar valores faltantes numéricos usando la mediana de cada columna.
for col in numeric_cols:
    X[col] = X[col].fillna(X[col].median())

# 9. Rellenar valores faltantes categóricos usando el texto "Missing".
for col in categorical_cols:
    X[col] = X[col].fillna("Missing")



In [5]:
# 10. Convertir variables categóricas en variables dummy para que la regresión lineal pueda usarlas.
X_encoded = pd.get_dummies(X, drop_first=True)


In [6]:
# 11. Dividir los datos en entrenamiento y prueba después de preparar las variables.
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42
)


In [7]:
# 12. Crear el modelo de Regresión Lineal.
model = LinearRegression()

# 13. Entrenar el modelo usando únicamente los datos de entrenamiento.
model.fit(X_train, y_train)

# 14. Generar predicciones sobre el conjunto de prueba.
y_pred = model.predict(X_test)


In [9]:
# 15. Evaluar el modelo usando MAE, RMSE y R² para entender qué tan bien predice.
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

# Imprimir las métricas principales del modelo.
print("\nResultados del modelo:")
print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²:", round(r2, 4))



Resultados del modelo:
MAE: 0.5
RMSE: 0.62
R²: 0.4032


In [10]:
# 16 - Estadístico F y P-values (Statsmodels)
# Asegurar que X_train sea completamente numérico antes de añadir la constante
X_train_numeric = X_train.astype(float)

# Añadir una constante (intercepto) a las variables independientes para statsmodels
X_train_sm = sm.add_constant(X_train_numeric)

# Crear y ajustar el modelo OLS
model_sm = sm.OLS(y_train, X_train_sm)
results_sm = model_sm.fit()

# Imprimir el resumen de los resultados de la regresión
print("\nEstadísticas del modelo (Statsmodels):")
print(results_sm.summary())



Estadísticas del modelo (Statsmodels):
                            OLS Regression Results                            
Dep. Variable:                quality   R-squared:                       0.348
Model:                            OLS   Adj. R-squared:                  0.342
Method:                 Least Squares   F-statistic:                     61.48
Date:                Thu, 04 Jun 2026   Prob (F-statistic):          1.48e-109
Time:                        02:21:36   Log-Likelihood:                -1266.4
No. Observations:                1279   AIC:                             2557.
Df Residuals:                    1267   BIC:                             2619.
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------

In [18]:
# 17. Visualizar valores reales vs predichos para evaluar gráficamente el desempeño del modelo.
# (Se guarda el archivo directamente sin usar .show() para cumplir los estándares del entorno)
plt.scatter(y_test, y_pred, alpha=0.6)
plt.xlabel("Calidad real")
plt.ylabel("Calidad predicha")
plt.title("Regresión Lineal: Calidad real vs Calidad predicha")

# Línea de referencia perfecta.
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()])
plt.close()

In [19]:
# 18. Mostrar algunos ejemplos comparando precios reales y predichos.
results = pd.DataFrame({
    "Calidad real": y_test.values,
    "Calidad predicho": y_pred
})

print("\nPrimeras predicciones:")
print(results.head(10))


Primeras predicciones:
   Calidad real  Calidad predicho
0             6          5.346664
1             5          5.056313
2             6          5.664470
3             5          5.464515
4             6          5.725185
5             5          5.279287
6             5          5.034217
7             5          5.126233
8             5          5.745343
9             6          5.686650
